## EDA

### General Setup Things

In [1]:
# Initial setup

import sqlite3
import pandas as pd
pd.set_option('display.max_columns', None)
con = sqlite3.connect("n1_data_ops_challenge.db")
cur = con.cursor()

In [2]:
# Finding table names

res = cur.execute("""
    SELECT name FROM sqlite_schema
    WHERE type="table" AND name NOT LIKE 'sqlite_&';
""")
table_names = res.fetchall()
print(table_names)

[('model_scores_by_zip',), ('roster_1',), ('roster_2',), ('roster_3',), ('roster_4',), ('roster_5',)]


| Schema Names |
| ------------ |
| model_scores_by_zip |
| roster_1 |
| roster_2 |
| roster_3 |
| roster_4 |
| roster_5 |

### Model Scores Exploration

In [3]:
# Load model_scores

sql_query = """
    SELECT * FROM model_scores_by_zip;
"""
model_scores = pd.read_sql_query(sql_query, con)
model_scores.head()

,zcta,state_code,state name,neighborhood_stress_score,algorex_sdoh_composite_score,social_isolation_score,transportation_access_score,food_access_score,unstable_housing_score,state_govt_assistance,homeless_indicator,derived_indicator
0,94720,6.0,California,-0.31,6.19,2.37,3.89,3.06,2.03,0.59,10.0,1
1,95675,6.0,California,-0.31,6.18,3.75,4.42,3.90,2.76,0.33,6.0,1
2,95699,6.0,California,-0.31,6.18,3.75,4.42,3.90,2.76,0.33,6.0,1
3,95930,6.0,California,0.33,6.00,3.47,3.95,3.01,3.05,0.76,10.0,1
4,95941,6.0,California,0.33,6.00,3.47,3.95,3.01,3.05,0.76,10.0,1


In [4]:
# Checking types in sql

sql_query = """
    SELECT sql FROM sqlite_schema
    WHERE name='model_scores_by_zip';
"""
model_scores_create_query = pd.read_sql_query(sql_query, con)
print(model_scores_create_query.iloc[0,0])

CREATE TABLE model_scores_by_zip(
  zcta INT,
  state_code REAL,
  "state name" TEXT,
  neighborhood_stress_score REAL,
  algorex_sdoh_composite_score REAL,
  social_isolation_score REAL,
  transportation_access_score REAL,
  food_access_score REAL,
  unstable_housing_score REAL,
  state_govt_assistance REAL,
  homeless_indicator REAL,
  derived_indicator INT
)


| **Column** | **Data Type** |
| ------ | --------- |
| zcta | INT |
| state_code | REAL |
| "state name" | TEXT |
| neighborhood_stress_score | REAL |
| algorex_sdoh_composite_score | REAL |
| social_isolation_score | REAL |
| transportation_access_score | REAL |
| food_access_score | REAL |
| unstable_housing_score | REAL |
| state_govt_assistance | REAL |
| homeless_indicator | REAL |
| derived_indicator | INT |

In [5]:
# Checking types in python

model_scores.dtypes

zcta                              int64
state_code                      float64
state name                       object
neighborhood_stress_score       float64
algorex_sdoh_composite_score    float64
social_isolation_score          float64
transportation_access_score     float64
food_access_score               float64
unstable_housing_score          float64
state_govt_assistance           float64
homeless_indicator              float64
derived_indicator                 int64
dtype: object

| **Column** | **Data Type** |
| ------ | --------- |
| zcta | int64|
| state_code | float64 |
| state name | str |
| neighborhood_stress_score | float64 |
| algorex_sdoh_composite_score | float64 |
| social_isolation_score | float64 |
| transportation_access_score | float64 |
| food_access_score | float64 |
| unstable_housing_score | float64 |
| state_govt_assistance | float64 |
| homeless_indicator | float64 |
| derived_indicator | int64 |

Looks like all the data types make sense. Next to check null counts etc.

In [6]:
model_scores.isnull().sum()

zcta                            0
state_code                      0
state name                      0
neighborhood_stress_score       0
algorex_sdoh_composite_score    0
social_isolation_score          0
transportation_access_score     0
food_access_score               0
unstable_housing_score          0
state_govt_assistance           0
homeless_indicator              0
derived_indicator               0
dtype: int64

In [7]:
model_scores.describe()

,zcta,state_code,neighborhood_stress_score,algorex_sdoh_composite_score,social_isolation_score,transportation_access_score,food_access_score,unstable_housing_score,state_govt_assistance,homeless_indicator,derived_indicator
count,1760.000000,1760.0,1760.000000,1760.000000,1760.000000,1760.000000,1760.000000,1760.000000,1760.000000,1760.000000,1760.000000
mean,93662.508523,6.0,-0.062892,6.427369,3.071813,4.238028,3.159642,2.638472,0.586636,7.162500,0.094886
std,1816.789712,0.0,0.690468,0.461369,0.803491,0.589008,0.801018,1.075871,0.541709,3.006486,0.293141
min,90001.000000,6.0,-1.410000,4.300000,0.000000,2.710000,0.280000,0.200000,0.000000,2.000000,0.000000
25%,92257.750000,6.0,-0.550000,6.117500,2.450000,3.880000,2.730000,1.970000,0.250000,4.000000,0.000000
50%,93656.500000,6.0,-0.210000,6.370000,2.920000,4.180000,3.210000,2.610000,0.440000,8.000000,0.000000
75%,95380.500000,6.0,0.290000,6.670000,3.600000,4.490000,3.700000,3.090000,0.770000,10.000000,0.000000
max,96161.000000,6.0,6.120000,8.770000,8.330000,8.750000,6.900000,7.880000,7.500000,10.000000,1.000000


Will probably use zcta as primary key for joins. Neighborhood level granularity for only the state of California.

### Roster 1 Exploration

In [8]:
# Load model_scores

sql_query = """
    SELECT * FROM roster_1;
"""
roster_1 = pd.read_sql_query(sql_query, con)
roster_1.head()

,Person_Id,First_Name,Last_Name,Dob,Age,Gender,Street_Address,State,City,Zip,eligibility_start_date,eligibility_end_date,payer
0,15340001,Daniel,Smith,2017-04-27,5,Male,1505 Alvarez Spur Suite 902,California,Lake Sharonburgh,93546,2021-08-01,2021-11-01,Madv
1,15340006,Todd,Austin,1934-01-06,88,Male,4731 Howe Ridge,California,New Rachel,95451,2021-08-01,2023-08-01,Madv
2,15340022,Leroy,Wilson,1960-09-20,62,Male,9710 Brianna Trail Apt. 145,California,Port Meredith,92222,2021-08-01,2024-01-01,Mdcd
3,15340042,Monica,Elmquist,1981-09-02,41,Female,47630 Sampson Throughway Suite 673,California,North Desireetown,95471,2021-08-01,2025-10-01,Mdcd
4,15340052,Betty,Read,1977-09-23,45,Female,78146 Angelica Lights Suite 526,California,Williambury,95018,2021-08-01,2024-08-01,Madv


In [9]:
# Checking types in sql

sql_query = """
    SELECT sql FROM sqlite_schema
    WHERE name='roster_1';
"""
roster_1_create_query = pd.read_sql_query(sql_query, con)
print(roster_1_create_query.iloc[0,0])

CREATE TABLE roster_1(
  Person_Id,
  First_Name,
  Last_Name,
  Dob,
  Age,
  Gender,
  Street_Address,
  State,
  City,
  Zip,
  eligibility_start_date,
  eligibility_end_date,
  payer
)


No column types were declared in the schema (SQLite allows this — columns default to no type affinity), so every value is stored and read back as text, even ones that look numeric like `Age` and `Zip`.

In [10]:
# Checking types in python

roster_1.dtypes

Person_Id                 object
First_Name                object
Last_Name                 object
Dob                       object
Age                       object
Gender                    object
Street_Address            object
State                     object
City                      object
Zip                       object
eligibility_start_date    object
eligibility_end_date      object
payer                     object
dtype: object

In [11]:
roster_1.isnull().sum()

Person_Id                 0
First_Name                0
Last_Name                 0
Dob                       0
Age                       0
Gender                    0
Street_Address            0
State                     0
City                      0
Zip                       0
eligibility_start_date    0
eligibility_end_date      0
payer                     0
dtype: int64

In [12]:
roster_1.describe()

,Person_Id,First_Name,Last_Name,Dob,Age,Gender,Street_Address,State,City,Zip,eligibility_start_date,eligibility_end_date,payer
count,23659,23659,23659,23659,23659,23659,23659,23659,23659,23659,23659,23659,23659
unique,23659,2601,9691,17453,101,2,23659,1,14990,1760,2,58,2
top,15340001,John,Smith,2011-07-14,43,Male,1505 Alvarez Spur Suite 902,California,West Michael,90620,2021-08-01,2022-06-01,Mdcd
freq,1,486,225,5,275,12006,1,23659,24,27,12011,456,14726


In [13]:
# Checking categorical values

for col in ['Gender', 'State', 'payer']:
    print(col, roster_1[col].unique())

Gender ['Male' 'Female']
State ['California']
payer ['Madv' 'Mdcd']


No nulls. `Gender` is Male/Female, `State` is always "California", `payer` is `Mdcd` (Medicaid) or `Madv` (Medicare Advantage). `Dob`/`eligibility_start_date`/`eligibility_end_date` are in `YYYY-MM-DD` format. `Person_Id` and `Zip` are 8-digit and 5-digit numeric strings respectively, with no duplicate `Person_Id` within this table.

### Roster 2 Exploration

In [14]:
# Load roster_2

sql_query = """
    SELECT * FROM roster_2;
"""
roster_2 = pd.read_sql_query(sql_query, con)
roster_2.head()

,Person_Id,First_Name,Last_Name,Dob,Age,Gender,Street_Address,State,City,Zip,eligibility_start_date,eligibility_end_date,payer
0,15340005,Maritza,Castellana,02/09/1979,43,Female,4097 Johnny Road,California,East Carolyntown,93206,10/01/2021,02/01/2023,Madv
1,15340011,Cynthia,Baker,07/12/1959,63,Female,55537 Ramos Drive,California,Kimberlymouth,95838,10/01/2021,10/01/2022,Madv
2,15340071,Dee,Silva,03/04/1989,33,Male,36858 Bates Flats Apt. 277,California,Stephenhaven,91501,10/01/2021,11/01/2022,Mdcd
3,15340092,Betty,Elmore,06/09/1962,60,Female,841 Bradford Heights Apt. 459,California,Greenmouth,90094,10/01/2021,06/01/2023,Mdcd
4,15340096,Lana,Gamble,05/03/2002,20,Female,3391 Emily Underpass,California,Chadburgh,92060,10/01/2021,02/01/2025,Mdcd


In [15]:
roster_2.isnull().sum()

Person_Id                 0
First_Name                0
Last_Name                 0
Dob                       0
Age                       0
Gender                    0
Street_Address            0
State                     0
City                      0
Zip                       0
eligibility_start_date    0
eligibility_end_date      0
payer                     0
dtype: int64

In [16]:
# Checking categorical values and date formats

for col in ['Gender', 'State', 'payer']:
    print(col, roster_2[col].unique())
print('Dob sample:', roster_2['Dob'].head(3).tolist())
print('eligibility_start_date sample:', roster_2['eligibility_start_date'].head(3).tolist())

Gender ['Female' 'Male']
State ['California']
payer ['Madv' 'Mdcd']
Dob sample: ['02/09/1979', '07/12/1959', '03/04/1989']
eligibility_start_date sample: ['10/01/2021', '10/01/2021', '10/01/2021']


**Data quality issue:** unlike `roster_1`, the date columns in `roster_2` (`Dob`, `eligibility_start_date`, `eligibility_end_date`) are formatted as `MM/DD/YYYY` instead of `YYYY-MM-DD`. `Gender`, `State`, and `payer` values are otherwise consistent with `roster_1`, and there are no nulls or in-table duplicate `Person_Id`s.

In [17]:
# Confirming MM/DD/YYYY (not DD/MM/YYYY) for roster_2's date columns, rather than assuming it

for col in ['Dob', 'eligibility_start_date', 'eligibility_end_date']:
    second_token = roster_2[col].str.split('/').str[1].astype(int)
    print(col, '- rows where the middle token is > 12:', (second_token > 12).sum(), '| max middle token:', second_token.max())

Dob - rows where the middle token is > 12: 14088 | max middle token: 31
eligibility_start_date - rows where the middle token is > 12: 0 | max middle token: 1
eligibility_end_date - rows where the middle token is > 12: 0 | max middle token: 1


Since neither token in a `MM/DD/YYYY` (or `DD/MM/YYYY`) string ever exceeds 12 by itself, magnitude alone can't prove which slot is the month — *unless* the other slot happens to exceed 12 somewhere, which would rule out that slot being a month.

For `Dob`, that happens: 14,088 of 23,392 rows have a middle token above 12 (up to 31), so that slot can only be the day — confirming `Dob` is `MM/DD/YYYY`, not `DD/MM/YYYY`.

`eligibility_start_date`/`eligibility_end_date` don't get the same direct proof — every eligibility date lands on the 1st of a month, so the middle token there is always `01` and can't rule anything out by itself. But since `Dob` in this same extract is confirmed month-first, and the other four rosters all show eligibility dates as `YYYY-MM-01` (always first-of-month), it's consistent to read `roster_2`'s `10/01/2021` as `Oct 1, 2021` rather than `Jan 10, 2021` — which also matches the Oct-Nov 2021 enrollment window `roster_2` occupies among the other rosters (see the eligibility date range check below). So `MM/DD/YYYY` for all three date columns is confirmed by direct evidence (`Dob`) plus consistency (eligibility dates), not just assumed from US convention.

### Roster 3 Exploration

In [18]:
# Load roster_3

sql_query = """
    SELECT * FROM roster_3;
"""
roster_3 = pd.read_sql_query(sql_query, con)
roster_3.head()

,Person_Id,First_Name,Last_Name,Dob,Age,Gender,Street_Address,State,City,Zip,eligibility_start_date,eligibility_end_date,payer
0,15340053,Nathaniel,Sharkey,1925-01-26,97,Male,92776 Charles Lights Suite 296,California,East Veronica,95461,2021-12-01,2022-01-01,Mdcd
1,15340064,Lola,Porterfield,1959-11-10,63,Female,3584 Larry Crest,California,Port Katrinaborough,92243,2021-12-01,2024-08-01,Madv
2,15340076,Gina,Stovall,1950-07-18,72,Female,4789 Petersen Light Apt. 391,California,Amandaland,91902,2021-12-01,2023-08-01,Mdcd
3,15340085,Franklin,Hillier,2022-06-18,0,Male,794 Brian Park,California,Johnsonfort,94306,2021-12-01,2025-03-01,Mdcd
4,15340106,Leona,Leesman,2000-05-14,22,Female,936 Robert Circle Suite 457,California,South Ericberg,94512,2021-12-01,2024-08-01,Madv


In [19]:
roster_3.isnull().sum()

Person_Id                 0
First_Name                0
Last_Name                 0
Dob                       0
Age                       0
Gender                    0
Street_Address            0
State                     0
City                      0
Zip                       0
eligibility_start_date    0
eligibility_end_date      0
payer                     0
dtype: int64

In [20]:
# Checking categorical values and date formats

for col in ['Gender', 'State', 'payer']:
    print(col, roster_3[col].unique())
print('Dob sample:', roster_3['Dob'].head(3).tolist())
print('eligibility_start_date sample:', roster_3['eligibility_start_date'].head(3).tolist())

Gender ['Male' 'Female']
State ['California']
payer ['Mdcd' 'Madv']
Dob sample: ['1925-01-26', '1959-11-10', '1950-07-18']
eligibility_start_date sample: ['2021-12-01', '2021-12-01', '2021-12-01']


`roster_3` is back to the `YYYY-MM-DD` date format from `roster_1`, with consistent `Gender`/`State`/`payer` values, no nulls, and no in-table duplicate `Person_Id`s. This is also the largest roster (34,951 rows).

### Roster 4 Exploration

In [21]:
# Load roster_4

sql_query = """
    SELECT * FROM roster_4;
"""
roster_4 = pd.read_sql_query(sql_query, con)
roster_4.head()

,Person_Id,First_Name,Last_Name,Dob,Age,Gender,Street_Address,State,City,Zip,eligibility_start_date,eligibility_end_date,payer
0,15340034,Amber,Smith,2000-02-01,22,Female,404 Gardner Pike Suite 348,CA,North Jefferyport,95620,2022-02-01,2022-06-01,Mdcd
1,15340089,Victor,Marble,1947-02-15,75,Male,63901 Jeffrey Extensions Suite 767,CA,West Lisa,93626,2022-02-01,2025-10-01,Mdcd
2,15340093,Edward,Quarry,2010-11-16,12,Male,30703 Jared Ville Suite 612,CA,Andrewbury,95387,2022-02-01,2022-12-01,Mdcd
3,15340097,Elinor,Rose,1928-06-26,94,Female,916 Ryan Springs,CA,Christopherberg,92404,2022-02-01,2022-06-01,Mdcd
4,15340103,Gregory,Allen,2020-03-05,2,Male,0893 Joshua Keys Suite 905,CA,North Allenmouth,95681,2022-02-01,2026-01-01,Mdcd


In [22]:
roster_4.isnull().sum()

Person_Id                 0
First_Name                0
Last_Name                 0
Dob                       0
Age                       0
Gender                    0
Street_Address            0
State                     0
City                      0
Zip                       0
eligibility_start_date    0
eligibility_end_date      0
payer                     0
dtype: int64

In [23]:
# Checking categorical values and date formats

for col in ['Gender', 'State', 'payer']:
    print(col, roster_4[col].unique())
print('Dob sample:', roster_4['Dob'].head(3).tolist())
print('eligibility_start_date sample:', roster_4['eligibility_start_date'].head(3).tolist())

Gender ['Female' 'Male']
State ['CA']
payer ['Mdcd' 'Madv']
Dob sample: ['2000-02-01', '1947-02-15', '2010-11-16']
eligibility_start_date sample: ['2022-02-01', '2022-02-01', '2022-02-01']


**Data quality issue:** `roster_4` uses the `YYYY-MM-DD` date format, but `State` is abbreviated as `"CA"` instead of the full `"California"` used everywhere else. Will need to standardize this before combining rosters or joining on state.

### Roster 5 Exploration

In [24]:
# Load roster_5

sql_query = """
    SELECT * FROM roster_5;
"""
roster_5 = pd.read_sql_query(sql_query, con)
roster_5.head()

,Person_Id,First_Name,Last_Name,Gender,Dob,Age,Street_Address,State,City,Zip,payer,eligibility_start_date,eligibility_end_date
0,15340012,Billy,Pacifico,Male,1989-04-03,33,6176 Nicholas Turnpike Apt. 850,California,West Dorothyburgh,90047,Mdcd,2022-04-01,2026-03-01
1,15340013,John,Celis,Male,1985-01-07,37,12708 Morgan Parks,California,West Jonstad,92831,Madv,2022-04-01,2023-08-01
2,15340038,Rose,Shaffer,Female,1929-08-10,93,349 Clark Light Suite 153,California,Jenniferstad,95536,Madv,2022-04-01,2022-10-01
3,15340041,Edith,Larson,Female,1938-08-29,83,93705 Brittany Manor Apt. 126,California,Stonechester,95835,Mdcd,2022-04-01,2023-01-01
4,15340046,Shaun,Hughes,Male,2020-06-18,2,6549 Jesse Harbors,California,Grosshaven,90604,Madv,2022-04-01,2025-05-01


In [25]:
roster_5.isnull().sum()

Person_Id                 0
First_Name                0
Last_Name                 0
Gender                    0
Dob                       0
Age                       0
Street_Address            0
State                     0
City                      0
Zip                       0
payer                     0
eligibility_start_date    0
eligibility_end_date      0
dtype: int64

In [26]:
# Checking categorical values and date formats

for col in ['Gender', 'State', 'payer']:
    print(col, roster_5[col].unique())
print('Dob sample:', roster_5['Dob'].head(3).tolist())
print('eligibility_start_date sample:', roster_5['eligibility_start_date'].head(3).tolist())

Gender ['Male' 'Female']
State ['California']
payer ['Mdcd' 'Madv']
Dob sample: ['1989-04-03', '1985-01-07', '1929-08-10']
eligibility_start_date sample: ['2022-04-01', '2022-04-01', '2022-04-01']


`roster_5` uses `YYYY-MM-DD` dates and full `"California"` state names, consistent with `roster_1`/`roster_3`. Note the column order also differs slightly from the other four rosters (`Gender` and `Dob` are swapped, and `payer` comes before the eligibility dates) — this doesn't matter once loaded into a DataFrame, but is worth knowing if reading raw rows. This is the largest roster (37,403 rows), with no nulls or in-table duplicate `Person_Id`s.

### Cross-Roster Exploration

Each roster looks like a standalone extract, but they may represent overlapping enrollment periods for the same underlying population. Checking for `Person_Id` overlap and normalizing formats before combining them.

In [27]:
# Checking Person_Id overlap between every pair of rosters

rosters = {
    'roster_1': roster_1,
    'roster_2': roster_2,
    'roster_3': roster_3,
    'roster_4': roster_4,
    'roster_5': roster_5,
}
id_sets = {name: set(df['Person_Id']) for name, df in rosters.items()}
names = list(id_sets.keys())

overlap = pd.DataFrame(index=names, columns=names, dtype=int)
for a in names:
    for b in names:
        if a == b:
            overlap.loc[a, b] = len(id_sets[a])
        else:
            shared_ids = id_sets[a] & id_sets[b]
            overlap.loc[a, b] = len(shared_ids)
overlap

,roster_1,roster_2,roster_3,roster_4,roster_5
roster_1,23659.0,0.0,0.0,0.0,0.0
roster_2,0.0,23392.0,0.0,0.0,13775.0
roster_3,0.0,0.0,34951.0,10845.0,0.0
roster_4,0.0,0.0,10845.0,22900.0,0.0
roster_5,0.0,13775.0,0.0,0.0,37403.0


Only two pairs overlap: `roster_2`/`roster_5` share 13,775 `Person_Id`s, and `roster_3`/`roster_4` share 10,845. Every other pair has zero overlap. Next, checking whether the overlapping records are true duplicates (identical rows) or the same person with different data (e.g. a changed eligibility period).

In [28]:
# Normalizing formats so rosters can be compared apples-to-apples

def normalize(df):
    df = df.copy()
    df['Dob'] = pd.to_datetime(df['Dob'], format='mixed').dt.strftime('%Y-%m-%d')
    df['eligibility_start_date'] = pd.to_datetime(df['eligibility_start_date'], format='mixed').dt.strftime('%Y-%m-%d')
    df['eligibility_end_date'] = pd.to_datetime(df['eligibility_end_date'], format='mixed').dt.strftime('%Y-%m-%d')
    df['State'] = df['State'].replace({'CA': 'California'})
    return df

normalized = {name: normalize(df) for name, df in rosters.items()}

# Comparing the roster_2 / roster_5 overlap row-by-row on every shared column
compare_cols = ['First_Name', 'Last_Name', 'Dob', 'Age', 'Gender', 'Street_Address',
                 'State', 'City', 'Zip', 'eligibility_start_date', 'eligibility_end_date', 'payer']
merged_2_5 = normalized['roster_2'].merge(normalized['roster_5'], on='Person_Id', suffixes=('_2', '_5'))
mismatches_2_5 = {col: (merged_2_5[f'{col}_2'] != merged_2_5[f'{col}_5']).sum() for col in compare_cols}
print('roster_2 vs roster_5 mismatches per column:', mismatches_2_5)

merged_3_4 = normalized['roster_3'].merge(normalized['roster_4'], on='Person_Id', suffixes=('_3', '_4'))
mismatches_3_4 = {col: (merged_3_4[f'{col}_3'] != merged_3_4[f'{col}_4']).sum() for col in compare_cols}
print('roster_3 vs roster_4 mismatches per column:', mismatches_3_4)

roster_2 vs roster_5 mismatches per column: {'First_Name': np.int64(0), 'Last_Name': np.int64(0), 'Dob': np.int64(0), 'Age': np.int64(0), 'Gender': np.int64(0), 'Street_Address': np.int64(0), 'State': np.int64(0), 'City': np.int64(0), 'Zip': np.int64(0), 'eligibility_start_date': np.int64(0), 'eligibility_end_date': np.int64(0), 'payer': np.int64(0)}
roster_3 vs roster_4 mismatches per column: {'First_Name': np.int64(0), 'Last_Name': np.int64(0), 'Dob': np.int64(0), 'Age': np.int64(0), 'Gender': np.int64(0), 'Street_Address': np.int64(0), 'State': np.int64(0), 'City': np.int64(0), 'Zip': np.int64(0), 'eligibility_start_date': np.int64(0), 'eligibility_end_date': np.int64(0), 'payer': np.int64(0)}


Every shared column has zero mismatches in both pairs — the overlapping `Person_Id`s are **exact full-row duplicates** between `roster_2`/`roster_5` and between `roster_3`/`roster_4`, not the same person re-enrolled with different data. These 24,620 duplicate rows will need to be dropped when building a combined/deduplicated roster (e.g. `pd.concat(...).drop_duplicates(subset='Person_Id')`).

In [29]:
# Checking eligibility_start_date range per roster - are these sequential enrollment batches?

for name, df in normalized.items():
    es = pd.to_datetime(df['eligibility_start_date'])
    print(name, 'eligibility_start_date range:', es.min().date(), '-', es.max().date())

roster_1 eligibility_start_date range: 2021-08-01 - 2021-09-01
roster_2 eligibility_start_date range: 2021-10-01 - 2021-11-01
roster_3 eligibility_start_date range: 2021-12-01 - 2022-02-01
roster_4 eligibility_start_date range: 2022-02-01 - 2022-03-01
roster_5 eligibility_start_date range: 2021-10-01 - 2022-05-01


`roster_1` through `roster_4` each cover a roughly two-month enrollment window that steps forward in time (Aug 2021 → Mar 2022), consistent with sequential monthly/bi-monthly extracts. `roster_5` spans a much wider window (Oct 2021 → May 2022) that overlaps both `roster_2` and part of `roster_3`/`roster_4`'s ranges — which lines up with the exact-duplicate overlap found with `roster_2` above.

In [30]:
# Checking that every roster Zip has a matching zcta in model_scores_by_zip

zcta_set = set(model_scores['zcta'].astype(str))
for name, df in rosters.items():
    missing = set(df['Zip']) - zcta_set
    print(name, '- unique zips:', df['Zip'].nunique(), '| zips missing from model_scores_by_zip:', len(missing))

roster_1 - unique zips: 1760 | zips missing from model_scores_by_zip: 0
roster_2 - unique zips: 1760 | zips missing from model_scores_by_zip: 0
roster_3 - unique zips: 1760 | zips missing from model_scores_by_zip: 0
roster_4 - unique zips: 1760 | zips missing from model_scores_by_zip: 0
roster_5 - unique zips: 1760 | zips missing from model_scores_by_zip: 0


Every `Zip` value across all five rosters has a matching `zcta` in `model_scores_by_zip` (0 missing) — `Zip`/`zcta` is a clean, reliable join key between the rosters and the SDOH model scores, as long as `Zip` is cast to match `zcta`'s type (`str` vs `int`).

In [31]:
# Sanity-checking Age against Dob (as of a candidate snapshot date)

def age_asof(dob, ref):
    return ref.year - dob.year - ((ref.month, ref.day) < (dob.month, dob.day))

all_normalized = pd.concat(normalized.values(), ignore_index=True)
dob = pd.to_datetime(all_normalized['Dob'])
for ref_date in ['2022-06-23', '2022-12-31', '2023-01-01']:
    ref = pd.to_datetime(ref_date)
    computed = dob.apply(lambda d: age_asof(d, ref))
    mismatch = (computed.astype(str) != all_normalized['Age']).sum()
    print(f'as of {ref_date}: {mismatch} / {len(all_normalized)} rows mismatch Age')

as of 2022-06-23: 58794 / 142305 rows mismatch Age


as of 2022-12-31: 15624 / 142305 rows mismatch Age
as of 2023-01-01: 16003 / 142305 rows mismatch Age


**Data quality issue:** `Age` looks like it was computed from `Dob` as of a snapshot date around late 2022, but no single reference date reconciles all rows — the best candidate (2022-12-31) still leaves ~11% of rows mismatched. `Age` shouldn't be trusted as an always-accurate derived field; if age matters downstream, it's safer to (re)compute it from `Dob` against a clearly defined reference date rather than using the provided column as-is.

In [32]:
# Building a deduplicated master roster and joining it to model_scores_by_zip

master_roster = all_normalized.drop_duplicates(subset='Person_Id').reset_index(drop=True)
master_roster['Zip'] = master_roster['Zip'].astype(int)

joined = master_roster.merge(model_scores, left_on='Zip', right_on='zcta', how='left')
print('rows across all rosters:', len(all_normalized))
print('rows after de-duping on Person_Id:', len(master_roster))
print('rows with no matching model_scores_by_zip row:', joined['zcta'].isnull().sum())
joined.head()

rows across all rosters: 142305
rows after de-duping on Person_Id: 117685
rows with no matching model_scores_by_zip row: 0


,Person_Id,First_Name,Last_Name,Dob,Age,Gender,Street_Address,State,City,Zip,eligibility_start_date,eligibility_end_date,payer,zcta,state_code,state name,neighborhood_stress_score,algorex_sdoh_composite_score,social_isolation_score,transportation_access_score,food_access_score,unstable_housing_score,state_govt_assistance,homeless_indicator,derived_indicator
0,15340001,Daniel,Smith,2017-04-27,5,Male,1505 Alvarez Spur Suite 902,California,Lake Sharonburgh,93546,2021-08-01,2021-11-01,Madv,93546,6.0,California,-0.60,6.64,2.50,2.96,2.10,2.19,0.06,10.0,0
1,15340006,Todd,Austin,1934-01-06,88,Male,4731 Howe Ridge,California,New Rachel,95451,2021-08-01,2023-08-01,Madv,95451,6.0,California,-0.10,6.33,3.90,4.80,3.99,2.81,0.40,10.0,0
2,15340022,Leroy,Wilson,1960-09-20,62,Male,9710 Brianna Trail Apt. 145,California,Port Meredith,92222,2021-08-01,2024-01-01,Mdcd,92222,6.0,California,0.66,6.62,3.56,3.83,3.30,3.17,0.99,8.0,1
3,15340042,Monica,Elmquist,1981-09-02,41,Female,47630 Sampson Throughway Suite 673,California,North Desireetown,95471,2021-08-01,2025-10-01,Mdcd,95471,6.0,California,-0.43,6.40,2.65,3.95,2.88,2.06,0.33,10.0,1
4,15340052,Betty,Read,1977-09-23,45,Female,78146 Angelica Lights Suite 526,California,Williambury,95018,2021-08-01,2024-08-01,Madv,95018,6.0,California,-0.43,5.92,2.44,4.20,2.34,1.75,0.40,10.0,0


In [33]:
# Filtering down to people eligible at any point during 2025
# (i.e. their eligibility window overlaps [2025-01-01, 2025-12-31])

window_start = pd.Timestamp('2025-01-01')
window_end = pd.Timestamp('2025-12-31')

elig_start = pd.to_datetime(joined['eligibility_start_date'])
elig_end = pd.to_datetime(joined['eligibility_end_date'])

eligible_2025 = joined[(elig_start <= window_end) & (elig_end >= window_start)]
print('people eligible at some point in 2025:', len(eligible_2025), 'of', len(joined))
eligible_2025.head()

people eligible at some point in 2025: 39252 of 117685


,Person_Id,First_Name,Last_Name,Dob,Age,Gender,Street_Address,State,City,Zip,eligibility_start_date,eligibility_end_date,payer,zcta,state_code,state name,neighborhood_stress_score,algorex_sdoh_composite_score,social_isolation_score,transportation_access_score,food_access_score,unstable_housing_score,state_govt_assistance,homeless_indicator,derived_indicator
3,15340042,Monica,Elmquist,1981-09-02,41,Female,47630 Sampson Throughway Suite 673,California,North Desireetown,95471,2021-08-01,2025-10-01,Mdcd,95471,6.0,California,-0.43,6.40,2.65,3.95,2.88,2.06,0.33,10.0,1
5,15340055,Wilber,Williams,1960-03-15,62,Male,500 Perez Turnpike,California,West Julie,95922,2021-08-01,2025-05-01,Mdcd,95922,6.0,California,-0.72,6.81,2.50,4.70,4.55,2.61,0.08,10.0,0
12,15340256,Theresa,Quintero,1947-04-26,75,Female,870 Gomez Gateway,California,Burketown,95324,2021-08-01,2025-08-01,Madv,95324,6.0,California,-0.79,6.28,3.74,3.80,0.86,2.59,0.23,5.0,0
14,15340322,Kimberly,Lynch,2019-07-03,3,Female,66925 Reilly Light Suite 077,California,Michaelburgh,90049,2021-08-01,2026-04-01,Mdcd,90049,6.0,California,-0.60,7.11,2.85,4.18,2.49,1.13,0.15,10.0,0
20,15340477,Duane,Perez,1987-01-15,35,Male,329 Courtney Village,California,East Norman,92606,2021-08-01,2026-01-01,Madv,92606,6.0,California,-0.44,6.03,3.61,3.79,3.62,2.31,0.30,3.0,0


De-duping on `Person_Id` drops the expected 24,620 rows (142,305 → 117,685), and every remaining row joins cleanly to `model_scores_by_zip` with zero unmatched zips.

### Summary of Key Findings

- **6 tables**: `model_scores_by_zip` (1,760 California ZCTAs with SDOH scores) and 5 member rosters (`roster_1`-`roster_5`, 117,685 unique people after de-duping).
- **No nulls** anywhere across any table.
- **`Zip`/`zcta` is a clean join key** between rosters and `model_scores_by_zip` — 100% match rate, but types differ (`Zip` is a string, `zcta` is an int) so one side needs casting.
- **Exact duplicate rows across rosters**: `roster_2`/`roster_5` share 13,775 identical records and `roster_3`/`roster_4` share 10,845 — these look like overlapping-period extracts of the same underlying enrollment system rather than five independent populations. Must de-dupe on `Person_Id` (or the full row) before treating the rosters as one combined population.
- **Inconsistent formatting across rosters** that needs normalizing before combining:
  - Date columns (`Dob`, `eligibility_start_date`, `eligibility_end_date`) are `YYYY-MM-DD` in `roster_1`/`roster_3`/`roster_4`/`roster_5`, but `MM/DD/YYYY` in `roster_2`.
  - `State` is `"California"` everywhere except `roster_4`, which uses `"CA"`.
  - `roster_5`'s column order differs from the other four (harmless once loaded into a DataFrame by name).
- **No explicit SQL types** on any roster column — everything round-trips as text, so numeric-looking fields (`Age`, `Zip`, `Person_Id`) need explicit casting before numeric operations.
- **`Age` is not fully reliable** as a derived field — no single snapshot date reconciles it against `Dob` for all rows (best fit still ~11% mismatched); recompute from `Dob` if precise age is needed.
- Rosters appear to represent sequential ~2-month enrollment batches spanning Aug 2021 - mid 2022 (`roster_5` spans a wider window that overlaps several of the others), and `payer` is one of two values: `Mdcd` (Medicaid) or `Madv` (Medicare Advantage).

In [34]:
# making final df and creating table

std_member_info = pd.DataFrame()

new_cols_list = [
    'member_id', 'member_first_name',
    'member_last_name', 'date_of_birth',
    'main_address', 'city', 'state',
    'zip_code', 'payer',
    'eligibility_start_date', 'eligibility_end_date'
]
old_cols_list = [
    'Person_Id', 'First_Name', 'Last_Name',
    'Dob', 'Street_Address', 'City', 'State',
    'Zip', 'payer', 'eligibility_start_date',
    'eligibility_end_date'
]

for i in range(len(new_cols_list)):
    std_member_info[new_cols_list[i]] = eligible_2025[old_cols_list[i]]

std_member_info.head()

,member_id,member_first_name,member_last_name,date_of_birth,main_address,city,state,zip_code,payer,eligibility_start_date,eligibility_end_date
3,15340042,Monica,Elmquist,1981-09-02,47630 Sampson Throughway Suite 673,North Desireetown,California,95471,Mdcd,2021-08-01,2025-10-01
5,15340055,Wilber,Williams,1960-03-15,500 Perez Turnpike,West Julie,California,95922,Mdcd,2021-08-01,2025-05-01
12,15340256,Theresa,Quintero,1947-04-26,870 Gomez Gateway,Burketown,California,95324,Madv,2021-08-01,2025-08-01
14,15340322,Kimberly,Lynch,2019-07-03,66925 Reilly Light Suite 077,Michaelburgh,California,90049,Mdcd,2021-08-01,2026-04-01
20,15340477,Duane,Perez,1987-01-15,329 Courtney Village,East Norman,California,92606,Madv,2021-08-01,2026-01-01


In [35]:
# Writing std_member_info into the db as its own table

std_member_info.to_sql('std_member_info', con, if_exists='replace', index=False)

39252